# Telegram Cluster Group Amplification Pipeline

Notebook này đo amplification ở cấp **cluster group**, không phải từng file parquet/channel riêng lẻ.

Logic đúng theo yêu cầu:

```text
channel_1386429252.parquet
        ^^
        13 = cluster group

channel_1568241389.parquet
        ^^
        15 = cluster group
```

Pipeline:

```text
Hugging Face ZIP files
→ extract từng parquet tạm
→ lấy 2 số đầu sau dấu "_" làm cluster_key
→ gom tất cả parquet/channel cùng cluster_key
→ tạo cluster-level behavioral features
→ train RandomForest amplification model ở cấp cluster
→ export group_amplification_scores.csv
```

Output chính:

```text
/kaggle/working/group_amplification_scores.csv
/kaggle/working/amplification_model.joblib
/kaggle/working/telegram_cluster_amplification_outputs.zip
```


## 1. Cài thư viện


In [1]:
!pip -q install pyspark pyarrow scikit-learn joblib tqdm


## 2. Config


In [2]:

from pathlib import Path

HF_BASE = "https://huggingface.co/datasets/Tungtom2004/Telegram_politic_dataset/resolve/main"

ZIP_FILES = [
    "channels_10_parquet.zip",
    "channels_11_parquet.zip",
    "channels_12_parquet.zip",
    "channels_13_parquet.zip",
    "channels_14_parquet.zip",
    "channels_15_parquet.zip",
    "channels_16_parquet.zip",
    "channels_17_parquet.zip",
    "channels_18_parquet.zip",
    "channels_19_parquet.zip",
    "channels_20_parquet.zip",
    "channels_21_parquet.zip",
    "channels_22_parquet.zip",
    "channels_23_parquet.zip",
    "channels_24_parquet.zip",
]

# Test nhanh:
# MAX_ZIPS = 2
# MAX_PARQUET_PER_ZIP = 2
MAX_ZIPS = None
MAX_PARQUET_PER_ZIP = None

# 2 số đầu sau dấu "_" là cluster group.
CHANNEL_CLUSTER_DIGITS = 2

# Filter giống pipeline cũ
MIN_CONTENT_LENGTH = 30
MIN_FORWARDS = 3
MIN_TOXICITY = 0.2
ENGLISH_ONLY = True
POLITICAL_ONLY = True
SPAM_REGEX = r"(?i)(http|www|vip|deposit|register|bonus|usdt|airdrop|referral|withdrawal|claim|earn)"

# Target để tạo nhãn high amplification ở cấp cluster.
# Khuyến nghị:
# - forwards_per_channel: công bằng hơn giữa các cluster có số channel khác nhau
# - forwards_mean: mức lan truyền trung bình/message
# - forwards_sum: tổng amplification, nhưng dễ bias cluster lớn
TARGET_METRIC = "forwards_per_channel"
HIGH_QUANTILE = 0.75

LOW_THRESHOLD = 0.33
HIGH_THRESHOLD = 0.66

BASE = Path("/kaggle/working/data")
RAW = BASE / "raw_tmp"
TMP = BASE / "parquet_tmp"

PARTIAL_DIR = Path("/kaggle/working/_partial_cluster_features")
CLUSTER_FEATURES_OUTPUT = Path("/kaggle/working/cluster_features.parquet")
MODEL_OUTPUT = Path("/kaggle/working/amplification_model.joblib")
SCORES_OUTPUT = Path("/kaggle/working/group_amplification_scores.csv")
FAILED_OUTPUT = Path("/kaggle/working/failed_items.csv")
BUNDLE_OUTPUT = Path("/kaggle/working/telegram_cluster_amplification_outputs.zip")


## 3. Imports và Spark


In [3]:

import gc
import json
import os
import re
import shutil
import subprocess
import zipfile
from pathlib import Path

import joblib
import pandas as pd
from tqdm.auto import tqdm

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_spark():
    spark = (
        SparkSession.builder
        .appName("telegram-cluster-group-amplification")
        .config("spark.sql.shuffle.partitions", "32")
        .config("spark.driver.memory", "8g")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

spark = make_spark()
print("Spark ready")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 03:04:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready


## 4. Utils tải ZIP và tách cluster_key từ tên parquet


In [4]:

def free_gb(path="/kaggle/working"):
    return shutil.disk_usage(path).free / 1e9


def run_quiet(cmd):
    result = subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr[:3000])


def safe_remove_file(path):
    path = Path(path)
    if path.exists() and path.is_file():
        path.unlink()


def safe_remove_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)


def cleanup_temp():
    safe_remove_dir(TMP)
    TMP.mkdir(parents=True, exist_ok=True)
    spark.catalog.clearCache()
    gc.collect()


def download_zip(zip_name):
    RAW.mkdir(parents=True, exist_ok=True)
    zip_path = RAW / zip_name
    safe_remove_file(zip_path)

    url = f"{HF_BASE}/{zip_name}"
    print("Downloading:", url)
    run_quiet(["wget", "-O", str(zip_path), url])
    print(f"Downloaded {zip_path.name}: {zip_path.stat().st_size / 1e9:.2f} GB")
    return zip_path


def extract_one_parquet(zip_file, member, tmp_path):
    with zip_file.open(member) as src, open(tmp_path, "wb") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024 * 16)


def extract_cluster_parts_from_member(zip_name, member):
    """
    Từ tên parquet member, lấy:
    - channel_key: định danh file/channel đầy đủ
    - cluster_key: 2 số đầu sau dấu "_" để gom cluster group

    Ví dụ:
    channel_1386429252.parquet
    -> channel_key = channel_1386429252
    -> cluster_key = cluster_13
    """
    member_path = Path(member)
    parts = [member_path.stem] + [str(p) for p in member_path.parts[::-1]]

    # Pattern tốt nhất: channel_<digits dài>
    for part in parts:
        m = re.search(r"(channel_([0-9]{2,}))", part)
        if m:
            channel_key = m.group(1)
            digits = m.group(2)
            cluster_id = digits[:CHANNEL_CLUSTER_DIGITS]
            return channel_key, f"cluster_{cluster_id}", cluster_id

    # Pattern: channel_<2 số>_<phần còn lại>
    for part in parts:
        m = re.search(r"(channel_([0-9]{2})_[A-Za-z0-9_]+)", part)
        if m:
            channel_key = m.group(1)
            cluster_id = m.group(2)
            return channel_key, f"cluster_{cluster_id}", cluster_id

    # Nếu filename chỉ là số dài
    for part in parts:
        m = re.search(r"(?<!\d)(\d{6,})(?!\d)", part)
        if m:
            digits = m.group(1)
            channel_key = "channel_" + digits
            cluster_id = digits[:CHANNEL_CLUSTER_DIGITS]
            return channel_key, f"cluster_{cluster_id}", cluster_id

    # Fallback lấy từ folder/zip channels_15
    for part in parts + [zip_name]:
        m = re.search(r"channels?_([0-9]{2})", str(part))
        if m:
            cluster_id = m.group(1)
            fallback_channel = re.sub(r"[^a-zA-Z0-9_]+", "_", member_path.with_suffix("").as_posix()).strip("_")
            return fallback_channel, f"cluster_{cluster_id}", cluster_id

    fallback_channel = re.sub(r"[^a-zA-Z0-9_]+", "_", member_path.with_suffix("").as_posix()).strip("_")
    return fallback_channel or "unknown_channel", "cluster_unknown", "unknown"


print("Free disk:", f"{free_gb():.2f} GB")


Free disk: 20.94 GB


## 5. Feature engineering và partial aggregate ở cấp cluster


In [5]:

NUMERIC_COLS = [
    "political",
    "toxicity",
    "severe_toxicity",
    "identity_attack",
    "insult",
    "profanity",
    "threat",
    "forwards",
]

MODEL_FEATURE_COLUMNS = [
    "channel_count",
    "message_count",
    "messages_per_channel",
    "toxicity_mean",
    "toxicity_max",
    "toxicity_std",
    "severe_toxicity_mean",
    "severe_toxicity_max",
    "identity_attack_mean",
    "identity_attack_max",
    "insult_mean",
    "insult_max",
    "profanity_mean",
    "profanity_max",
    "threat_mean",
    "threat_max",
    "political_mean",
    "content_length_mean",
    "word_count_mean",
    "toxic_rate",
    "threat_rate",
    "severe_rate",
]


def ensure_columns(sdf):
    for c in NUMERIC_COLS:
        if c not in sdf.columns:
            sdf = sdf.withColumn(c, F.lit(0.0))
        sdf = sdf.withColumn(c, F.coalesce(F.col(c).cast("double"), F.lit(0.0)))

    if "content" not in sdf.columns:
        sdf = sdf.withColumn("content", F.lit(""))
    else:
        sdf = sdf.withColumn("content", F.coalesce(F.col("content").cast("string"), F.lit("")))

    if "language" not in sdf.columns:
        sdf = sdf.withColumn("language", F.lit("unknown"))

    return sdf


def filter_messages(sdf):
    sdf = sdf.filter(F.col("content").isNotNull())
    sdf = sdf.filter(F.length(F.col("content")) >= F.lit(MIN_CONTENT_LENGTH))

    if ENGLISH_ONLY:
        sdf = sdf.filter(F.lower(F.col("language").cast("string")) == F.lit("en"))

    if POLITICAL_ONLY:
        sdf = sdf.filter(F.col("political").cast("double") == F.lit(1.0))

    sdf = sdf.filter(F.col("toxicity").isNotNull())
    sdf = sdf.filter(F.col("toxicity") >= F.lit(float(MIN_TOXICITY)))

    sdf = sdf.filter(F.col("forwards").isNotNull())
    sdf = sdf.filter(F.col("forwards") >= F.lit(float(MIN_FORWARDS)))

    sdf = sdf.filter(~F.col("content").rlike(SPAM_REGEX))

    return sdf


def add_message_features(sdf):
    sdf = sdf.withColumn("content_length", F.length(F.col("content")))
    sdf = sdf.withColumn(
        "word_count",
        F.when(F.length(F.trim(F.col("content"))) == 0, F.lit(0))
         .otherwise(F.size(F.split(F.trim(F.col("content")), r"\s+")))
    )
    return sdf


def partial_cluster_aggregate(sdf):
    """
    Aggregate từng parquet thành partial cluster stats.
    Group key là __cluster_key, không phải __channel_key.
    """
    sdf = ensure_columns(sdf)
    sdf = filter_messages(sdf)
    sdf = add_message_features(sdf)

    return sdf.groupBy("__cluster_key").agg(
        F.first("__cluster_id", ignorenulls=True).alias("cluster_id"),
        F.collect_set("__channel_key").alias("channel_keys_set"),
        F.countDistinct("__channel_key").alias("channel_count_partial"),

        F.count(F.lit(1)).alias("message_count"),

        F.sum("toxicity").alias("toxicity_sum"),
        F.sum(F.col("toxicity") * F.col("toxicity")).alias("toxicity_sumsq"),
        F.max("toxicity").alias("toxicity_max"),

        F.sum("severe_toxicity").alias("severe_toxicity_sum"),
        F.max("severe_toxicity").alias("severe_toxicity_max"),

        F.sum("identity_attack").alias("identity_attack_sum"),
        F.max("identity_attack").alias("identity_attack_max"),

        F.sum("insult").alias("insult_sum"),
        F.max("insult").alias("insult_max"),

        F.sum("profanity").alias("profanity_sum"),
        F.max("profanity").alias("profanity_max"),

        F.sum("threat").alias("threat_sum"),
        F.max("threat").alias("threat_max"),

        F.sum("political").alias("political_sum"),

        F.sum("content_length").alias("content_length_sum"),
        F.sum("word_count").alias("word_count_sum"),

        F.sum("forwards").alias("forwards_sum"),
        F.max("forwards").alias("forwards_max"),

        F.sum(F.when(F.col("toxicity") >= 0.5, 1).otherwise(0)).alias("toxic_count"),
        F.sum(F.when(F.col("threat") >= 0.5, 1).otherwise(0)).alias("threat_count"),
        F.sum(F.when(F.col("severe_toxicity") >= 0.5, 1).otherwise(0)).alias("severe_count"),
    )


def combine_partial_cluster_features():
    partials_df = spark.read.parquet(str(PARTIAL_DIR / "*.parquet"))

    g = partials_df.groupBy("__cluster_key").agg(
        F.first("cluster_id", ignorenulls=True).alias("cluster_id"),
        F.array_distinct(F.flatten(F.collect_list("channel_keys_set"))).alias("channel_keys"),

        F.sum("message_count").alias("message_count"),

        F.sum("toxicity_sum").alias("toxicity_sum"),
        F.sum("toxicity_sumsq").alias("toxicity_sumsq"),
        F.max("toxicity_max").alias("toxicity_max"),

        F.sum("severe_toxicity_sum").alias("severe_toxicity_sum"),
        F.max("severe_toxicity_max").alias("severe_toxicity_max"),

        F.sum("identity_attack_sum").alias("identity_attack_sum"),
        F.max("identity_attack_max").alias("identity_attack_max"),

        F.sum("insult_sum").alias("insult_sum"),
        F.max("insult_max").alias("insult_max"),

        F.sum("profanity_sum").alias("profanity_sum"),
        F.max("profanity_max").alias("profanity_max"),

        F.sum("threat_sum").alias("threat_sum"),
        F.max("threat_max").alias("threat_max"),

        F.sum("political_sum").alias("political_sum"),

        F.sum("content_length_sum").alias("content_length_sum"),
        F.sum("word_count_sum").alias("word_count_sum"),

        F.sum("forwards_sum").alias("forwards_sum"),
        F.max("forwards_max").alias("forwards_max"),

        F.sum("toxic_count").alias("toxic_count"),
        F.sum("threat_count").alias("threat_count"),
        F.sum("severe_count").alias("severe_count"),
    )

    n = F.col("message_count")
    channel_count = F.size(F.col("channel_keys"))

    result = g.select(
        F.col("__cluster_key").alias("cluster_key"),
        F.col("cluster_id"),
        F.col("channel_keys"),
        F.slice(F.col("channel_keys"), 1, 20).alias("channel_keys_sample"),
        channel_count.alias("channel_count"),

        F.col("message_count"),
        (F.col("message_count") / F.greatest(channel_count, F.lit(1))).alias("messages_per_channel"),

        (F.col("toxicity_sum") / n).alias("toxicity_mean"),
        F.col("toxicity_max"),
        F.sqrt(
            F.greatest(
                (F.col("toxicity_sumsq") / n)
                - ((F.col("toxicity_sum") / n) * (F.col("toxicity_sum") / n)),
                F.lit(0.0),
            )
        ).alias("toxicity_std"),

        (F.col("severe_toxicity_sum") / n).alias("severe_toxicity_mean"),
        F.col("severe_toxicity_max"),

        (F.col("identity_attack_sum") / n).alias("identity_attack_mean"),
        F.col("identity_attack_max"),

        (F.col("insult_sum") / n).alias("insult_mean"),
        F.col("insult_max"),

        (F.col("profanity_sum") / n).alias("profanity_mean"),
        F.col("profanity_max"),

        (F.col("threat_sum") / n).alias("threat_mean"),
        F.col("threat_max"),

        (F.col("political_sum") / n).alias("political_mean"),

        (F.col("content_length_sum") / n).alias("content_length_mean"),
        (F.col("word_count_sum") / n).alias("word_count_mean"),

        F.col("forwards_sum"),
        (F.col("forwards_sum") / n).alias("forwards_mean"),
        F.col("forwards_max"),
        (F.col("forwards_sum") / F.greatest(channel_count, F.lit(1))).alias("forwards_per_channel"),

        (F.col("toxic_count") / n).alias("toxic_rate"),
        (F.col("threat_count") / n).alias("threat_rate"),
        (F.col("severe_count") / n).alias("severe_rate"),
    )

    return result.fillna(0)


## 6. Download ZIP từ Hugging Face và aggregate theo cluster group


In [6]:

safe_remove_dir(RAW)
safe_remove_dir(TMP)
safe_remove_dir(PARTIAL_DIR)
safe_remove_dir(CLUSTER_FEATURES_OUTPUT)

RAW.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

zip_files = ZIP_FILES[:MAX_ZIPS] if MAX_ZIPS is not None else ZIP_FILES

failed_items = []
part_idx = 0

for zip_i, zip_name in enumerate(tqdm(zip_files, desc="ZIP files", unit="zip"), start=1):
    zip_path = None

    try:
        if free_gb() < 4:
            raise RuntimeError(f"Low disk before download: {free_gb():.2f} GB")

        zip_path = download_zip(zip_name)

        with zipfile.ZipFile(zip_path, "r") as z:
            members = [
                m for m in z.namelist()
                if m.endswith(".parquet") and not m.endswith("/")
            ]

            if MAX_PARQUET_PER_ZIP is not None:
                members = members[:MAX_PARQUET_PER_ZIP]

            print(f"[{zip_i}/{len(zip_files)}] {zip_name} | parquet members: {len(members)}")

            for member_i, member in enumerate(tqdm(members, desc=zip_name, unit="parquet", leave=False), start=1):
                tmp_file = TMP / f"tmp_{zip_i}_{member_i}.parquet"

                try:
                    if free_gb() < 2:
                        raise RuntimeError(f"Low disk during extraction: {free_gb():.2f} GB")

                    channel_key, cluster_key, cluster_id = extract_cluster_parts_from_member(zip_name, member)

                    extract_one_parquet(z, member, tmp_file)
                    sdf = spark.read.parquet(str(tmp_file))

                    # Gắn metadata từ tên parquet.
                    # __cluster_key là khóa group chính.
                    sdf = (
                        sdf
                        .withColumn("__channel_key", F.lit(channel_key))
                        .withColumn("__cluster_key", F.lit(cluster_key))
                        .withColumn("__cluster_id", F.lit(cluster_id))
                        .withColumn("__source_zip", F.lit(zip_name))
                        .withColumn("__source_member", F.lit(member))
                    )

                    partial = partial_cluster_aggregate(sdf)

                    n_clusters = partial.count()
                    if n_clusters > 0:
                        part_idx += 1
                        out_part = PARTIAL_DIR / f"part_{part_idx:06d}.parquet"
                        partial.write.mode("overwrite").parquet(str(out_part))

                    safe_remove_file(tmp_file)
                    spark.catalog.clearCache()
                    gc.collect()

                except Exception as e:
                    failed_items.append((zip_name, member, str(e)[:500]))
                    safe_remove_file(tmp_file)
                    spark.catalog.clearCache()
                    gc.collect()
                    continue

        safe_remove_file(zip_path)
        cleanup_temp()

    except Exception as e:
        failed_items.append((zip_name, "__zip_level__", str(e)[:500]))
        if zip_path is not None:
            safe_remove_file(zip_path)
        cleanup_temp()
        continue

print("Partial files:", part_idx)

if failed_items:
    failed_df = pd.DataFrame(failed_items, columns=["zip_file", "member", "error"])
    failed_df.to_csv(FAILED_OUTPUT, index=False)
    display(failed_df.head(20))

if part_idx == 0:
    raise RuntimeError("Không tạo được partial cluster features nào.")


ZIP files:   0%|          | 0/15 [00:00<?, ?zip/s]

Downloading: https://huggingface.co/datasets/Tungtom2004/Telegram_politic_dataset/resolve/main/channels_10_parquet.zip
Downloaded channels_10_parquet.zip: 1.93 GB
[1/15] channels_10_parquet.zip | parquet members: 810


channels_10_parquet.zip:   0%|          | 0/810 [00:00<?, ?parquet/s]

Downloading: https://huggingface.co/datasets/Tungtom2004/Telegram_politic_dataset/resolve/main/channels_11_parquet.zip
Downloaded channels_11_parquet.zip: 6.08 GB
[2/15] channels_11_parquet.zip | parquet members: 2418


channels_11_parquet.zip:   0%|          | 0/2418 [00:00<?, ?parquet/s]

...

channels_23_parquet.zip:   0%|          | 0/49 [00:00<?, ?parquet/s]

Downloading: https://huggingface.co/datasets/Tungtom2004/Telegram_politic_dataset/resolve/main/channels_24_parquet.zip
Downloaded channels_24_parquet.zip: 0.00 GB
[15/15] channels_24_parquet.zip | parquet members: 36


channels_24_parquet.zip:   0%|          | 0/36 [00:00<?, ?parquet/s]

Partial files: 9827


,zip_file,member,error
0,channels_15_parquet.zip,channels_15_parquet/channel_1502533790.parquet,An error occurred while calling o4234140.parqu...
1,channels_18_parquet.zip,channel_1872306404.parquet,An error occurred while calling o7987328.parqu...
2,channels_18_parquet.zip,channel_1872375289.parquet,An error occurred while calling o8036861.parqu...
3,channels_18_parquet.zip,channel_1871959172.parquet,An error occurred while calling o8060681.parqu...
4,channels_18_parquet.zip,channel_1872509105.parquet,An error occurred while calling o8062590.parqu...
5,channels_18_parquet.zip,channel_1872694764.parquet,An error occurred while calling o8066409.parqu...
6,channels_18_parquet.zip,channel_1872361551.parquet,An error occurred while calling o8077212.parqu...
7,channels_18_parquet.zip,channel_1872223162.parquet,An error occurred while calling o8114021.parqu...
8,channels_18_parquet.zip,channel_1871060460.parquet,An error occurred while calling o8132121.parqu...
9,channels_18_parquet.zip,channel_1872726895.parquet,An error occurred while calling o8134030.parqu...


## 7. Combine thành bảng cluster-level features


In [7]:

cluster_features = combine_partial_cluster_features()

if CLUSTER_FEATURES_OUTPUT.exists():
    shutil.rmtree(CLUSTER_FEATURES_OUTPUT, ignore_errors=True)

cluster_features.write.mode("overwrite").parquet(str(CLUSTER_FEATURES_OUTPUT))

df_cluster = pd.read_parquet(CLUSTER_FEATURES_OUTPUT)

print("Cluster features shape:", df_cluster.shape)
display(df_cluster.head(30))

print("Cluster count:", df_cluster["cluster_key"].nunique())
print("Cluster distribution:")
display(
    df_cluster[["cluster_key", "channel_count", "message_count", "forwards_sum", "forwards_per_channel"]]
    .sort_values("cluster_key")
)


Cluster features shape: (14, 30)


,cluster_key,cluster_id,channel_keys,channel_keys_sample,channel_count,message_count,messages_per_channel,toxicity_mean,toxicity_max,toxicity_std,...,political_mean,content_length_mean,word_count_mean,forwards_sum,forwards_mean,forwards_max,forwards_per_channel,toxic_rate,threat_rate,severe_rate
0,cluster_10,10,"[channel_1095515118, channel_1088678000, chann...","[channel_1095515118, channel_1088678000, chann...",85,13487,158.670588,0.273088,0.96,0.074205,...,1.0,252.401498,37.841106,340999.0,25.283532,5067.0,4011.752941,0.017350,0.012012,0.000816
1,cluster_11,11,"[channel_1162933086, channel_1116064717, chann...","[channel_1162933086, channel_1116064717, chann...",592,86396,145.939189,0.335562,0.99,0.129576,...,1.0,331.020082,54.985034,11182480.0,129.432844,98340.0,18889.324324,0.107968,0.023728,0.004236
2,cluster_12,12,"[channel_1214535459, channel_1218420610, chann...","[channel_1214535459, channel_1218420610, chann...",951,140267,147.494217,0.338324,0.98,0.130703,...,1.0,349.558399,57.488754,12799581.0,91.251549,46009.0,13459.075710,0.111045,0.021994,0.003828
3,cluster_14,14,"[channel_1408988846, channel_1493281246, chann...","[channel_1408988846, channel_1493281246, chann...",1147,182620,159.215344,0.342063,0.99,0.134368,...,1.0,366.109906,60.817939,31414720.0,172.022341,97890.0,27388.596338,0.119034,0.022057,0.003499
4,cluster_15,15,"[channel_1500143050, channel_1500194629, chann...","[channel_1500143050, channel_1500194629, chann...",1226,181175,147.777325,0.341493,0.99,0.132617,...,1.0,378.251779,63.051464,36661042.0,202.351550,98340.0,29902.970636,0.116462,0.023160,0.004371
5,cluster_16,16,"[channel_1658763645, channel_1606432449, chann...","[channel_1658763645, channel_1606432449, chann...",1070,162705,152.060748,0.335960,0.99,0.131227,...,1.0,362.699782,59.771040,25483715.0,156.625273,98346.0,23816.556075,0.109609,0.022845,0.004757
6,cluster_18,18,"[channel_1800847669, channel_1808126025, chann...","[channel_1800847669, channel_1808126025, chann...",800,104610,130.762500,0.345614,0.99,0.144066,...,1.0,363.476417,60.438620,11815806.0,112.951018,96927.0,14769.757500,0.124520,0.027473,0.011892
7,cluster_19,19,"[channel_1927028788, channel_1964597198, chann...","[channel_1927028788, channel_1964597198, chann...",711,79929,112.417722,0.343001,0.99,0.137669,...,1.0,371.382752,61.063957,10130052.0,126.738130,97986.0,14247.611814,0.120457,0.024797,0.008808
8,cluster_20,20,"[channel_2015534729, channel_2031634478, chann...","[channel_2015534729, channel_2031634478, chann...",698,55043,78.858166,0.328110,0.99,0.127710,...,1.0,331.110150,54.766492,4382938.0,79.627528,97983.0,6279.280802,0.094908,0.024635,0.006286
9,cluster_21,21,"[channel_2136263556, channel_2140862011, chann...","[channel_2136263556, channel_2140862011, chann...",459,32256,70.274510,0.346426,0.99,0.145339,...,1.0,328.707093,54.111545,2685629.0,83.259828,26120.0,5851.043573,0.129061,0.031033,0.012401


Cluster count: 14
Cluster distribution:


,cluster_key,channel_count,message_count,forwards_sum,forwards_per_channel
0,cluster_10,85,13487,340999.0,4011.752941
1,cluster_11,592,86396,11182480.0,18889.324324
2,cluster_12,951,140267,12799581.0,13459.075710
12,cluster_13,970,156377,28451509.0,29331.452577
3,cluster_14,1147,182620,31414720.0,27388.596338
4,cluster_15,1226,181175,36661042.0,29902.970636
5,cluster_16,1070,162705,25483715.0,23816.556075
13,cluster_17,1002,136795,16340589.0,16307.973054
6,cluster_18,800,104610,11815806.0,14769.757500
7,cluster_19,711,79929,10130052.0,14247.611814


## 8. Train RandomForest ở cấp cluster group


In [8]:

def score_to_level(score):
    if score >= HIGH_THRESHOLD:
        return "High"
    if score >= LOW_THRESHOLD:
        return "Medium"
    return "Low"

cluster_df = pd.read_parquet(CLUSTER_FEATURES_OUTPUT)

if TARGET_METRIC not in cluster_df.columns:
    raise ValueError(
        f"TARGET_METRIC={TARGET_METRIC} không tồn tại. "
        "Hãy chọn forwards_per_channel, forwards_mean, forwards_sum, forwards_max."
    )

threshold = float(cluster_df[TARGET_METRIC].quantile(HIGH_QUANTILE))
cluster_df["high_amplification"] = (cluster_df[TARGET_METRIC] >= threshold).astype(int)

X = cluster_df[MODEL_FEATURE_COLUMNS]
y = cluster_df["high_amplification"]

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=1,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

metrics = {}

# Lưu ý: số cluster thường ít, nên evaluation chỉ mang tính demo.
if len(cluster_df) >= 10 and y.nunique() == 2:
    stratify = y if y.value_counts().min() >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=stratify,
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    metrics["accuracy"] = float(accuracy_score(y_test, pred))
    metrics["f1"] = float(f1_score(y_test, pred, zero_division=0))

    if len(set(y_test)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_test, proba))

    metrics["classification_report"] = classification_report(
        y_test,
        pred,
        zero_division=0,
        output_dict=True,
    )
else:
    model.fit(X, y)
    metrics["note"] = "Số cluster ít hoặc chỉ có một class. Train trên toàn bộ dữ liệu."

bundle = {
    "model": model,
    "feature_columns": MODEL_FEATURE_COLUMNS,
    "group_col": "cluster_key",
    "target_metric": TARGET_METRIC,
    "high_quantile": HIGH_QUANTILE,
    "target_threshold": threshold,
    "training_metrics": metrics,
    "note": "Model trained at cluster-group level, not channel/parquet level.",
}

joblib.dump(bundle, MODEL_OUTPUT)

# Model score
model_scores = model.predict_proba(X)[:, 1]

# Observed score: percentile rank theo TARGET_METRIC, hữu ích khi số cluster ít.
observed_scores = cluster_df[TARGET_METRIC].rank(pct=True).values

result = cluster_df.copy()
result["amplification_score"] = model_scores
result["observed_amplification_score"] = observed_scores
result["risk_level"] = result["amplification_score"].apply(score_to_level)
result["observed_risk_level"] = result["observed_amplification_score"].apply(score_to_level)

result = result.sort_values("amplification_score", ascending=False)

# CSV không nên lưu list quá dài dưới dạng object.
if "channel_keys" in result.columns:
    result["channel_keys"] = result["channel_keys"].apply(lambda x: ",".join(map(str, x)) if isinstance(x, (list, tuple)) else str(x))
if "channel_keys_sample" in result.columns:
    result["channel_keys_sample"] = result["channel_keys_sample"].apply(lambda x: ",".join(map(str, x)) if isinstance(x, (list, tuple)) else str(x))

result.to_csv(SCORES_OUTPUT, index=False)

print("Saved model:", MODEL_OUTPUT)
print("Saved scores:", SCORES_OUTPUT)
print("Target metric:", TARGET_METRIC)
print("Target threshold:", threshold)
print("Metrics:", metrics)

display(
    result[
        [
            "cluster_key",
            "cluster_id",
            "channel_count",
            "message_count",
            "messages_per_channel",
            "forwards_sum",
            "forwards_mean",
            "forwards_per_channel",
            "amplification_score",
            "observed_amplification_score",
            "risk_level",
            "observed_risk_level",
        ]
    ].head(30)
)


Saved model: /kaggle/working/amplification_model.joblib
Saved scores: /kaggle/working/group_amplification_scores.csv
Target metric: forwards_per_channel
Target threshold: 22584.748137155846
Metrics: {'accuracy': 0.75, 'f1': 0.0, 'roc_auc': 1.0, 'classification_report': {'0': {'precision': 0.75, 'recall': 1.0, 'f1-score': 0.8571428571428571, 'support': 3.0}, '1': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1.0}, 'accuracy': 0.75, 'macro avg': {'precision': 0.375, 'recall': 0.5, 'f1-score': 0.42857142857142855, 'support': 4.0}, 'weighted avg': {'precision': 0.5625, 'recall': 0.75, 'f1-score': 0.6428571428571428, 'support': 4.0}}}


,cluster_key,cluster_id,channel_count,message_count,messages_per_channel,forwards_sum,forwards_mean,forwards_per_channel,amplification_score,observed_amplification_score,risk_level,observed_risk_level
3,cluster_14,14,1147,182620,159.215344,31414720.0,172.022341,27388.596338,0.923333,0.857143,High,High
12,cluster_13,13,970,156377,161.213402,28451509.0,181.941775,29331.452577,0.846667,0.928571,High,High
4,cluster_15,15,1226,181175,147.777325,36661042.0,202.351550,29902.970636,0.820000,1.000000,High,High
5,cluster_16,16,1070,162705,152.060748,25483715.0,156.625273,23816.556075,0.413333,0.785714,Medium,High
6,cluster_18,18,800,104610,130.762500,11815806.0,112.951018,14769.757500,0.393333,0.571429,Medium,Medium
7,cluster_19,19,711,79929,112.417722,10130052.0,126.738130,14247.611814,0.380000,0.500000,Medium,Medium
9,cluster_21,21,459,32256,70.274510,2685629.0,83.259828,5851.043573,0.320000,0.285714,Low,Low
10,cluster_22,22,115,6049,52.600000,284376.0,47.012068,2472.834783,0.206667,0.142857,Low,Low
13,cluster_17,17,1002,136795,136.521956,16340589.0,119.453116,16307.973054,0.130000,0.642857,Low,Medium
2,cluster_12,12,951,140267,147.494217,12799581.0,91.251549,13459.075710,0.123333,0.428571,Low,Medium


## 9. Feature importance


In [9]:

try:
    clf = model.named_steps["clf"]
    importances = pd.DataFrame({
        "feature": MODEL_FEATURE_COLUMNS,
        "importance": clf.feature_importances_,
    }).sort_values("importance", ascending=False)

    display(importances)
except Exception as e:
    print("Cannot show feature importance:", e)


,feature,importance
1,message_count,0.148855
5,toxicity_std,0.144770
2,messages_per_channel,0.122124
0,channel_count,0.100169
8,identity_attack_mean,0.086015
3,toxicity_mean,0.060156
19,toxic_rate,0.058298
10,insult_mean,0.040736
18,word_count_mean,0.037957
9,identity_attack_max,0.031077


## 10. Đóng gói output để download


In [10]:

metadata_path = Path("/kaggle/working/training_metadata.json")

metadata = {
    "model_path": str(MODEL_OUTPUT),
    "scores_path": str(SCORES_OUTPUT),
    "cluster_features_path": str(CLUSTER_FEATURES_OUTPUT),
    "target_metric": TARGET_METRIC,
    "high_quantile": HIGH_QUANTILE,
    "target_threshold": threshold,
    "group_col": "cluster_key",
    "model_type": "RandomForestClassifier",
    "important_note": "Amplification is measured at cluster-group level. 2 digits after underscore define cluster_key.",
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

if BUNDLE_OUTPUT.exists():
    BUNDLE_OUTPUT.unlink()

with zipfile.ZipFile(BUNDLE_OUTPUT, "w", zipfile.ZIP_DEFLATED) as z:
    if MODEL_OUTPUT.exists():
        z.write(MODEL_OUTPUT, arcname="amplification_model.joblib")
    if SCORES_OUTPUT.exists():
        z.write(SCORES_OUTPUT, arcname="group_amplification_scores.csv")
    if metadata_path.exists():
        z.write(metadata_path, arcname="training_metadata.json")
    if FAILED_OUTPUT.exists():
        z.write(FAILED_OUTPUT, arcname="failed_items.csv")
    if CLUSTER_FEATURES_OUTPUT.exists() and CLUSTER_FEATURES_OUTPUT.is_dir():
        for p in CLUSTER_FEATURES_OUTPUT.rglob("*"):
            if p.is_file():
                z.write(p, arcname=str(Path("cluster_features.parquet") / p.relative_to(CLUSTER_FEATURES_OUTPUT)))

print("Saved bundle:", BUNDLE_OUTPUT)
print("Download this file:", BUNDLE_OUTPUT)


Saved bundle: /kaggle/working/telegram_cluster_amplification_outputs.zip
Download this file: /kaggle/working/telegram_cluster_amplification_outputs.zip


## 11. Kiểm tra nhanh output


In [11]:

df_scores = pd.read_csv(SCORES_OUTPUT)

print("Output shape:", df_scores.shape)
print("Unique cluster_key:", df_scores["cluster_key"].nunique())
display(df_scores.head(30))

print("Download:")
print(BUNDLE_OUTPUT)


Output shape: (14, 35)
Unique cluster_key: 14


,cluster_key,cluster_id,channel_keys,channel_keys_sample,channel_count,message_count,messages_per_channel,toxicity_mean,toxicity_max,toxicity_std,...,forwards_max,forwards_per_channel,toxic_rate,threat_rate,severe_rate,high_amplification,amplification_score,observed_amplification_score,risk_level,observed_risk_level
0,cluster_14,14,['channel_1408988846' 'channel_1493281246' 'ch...,['channel_1408988846' 'channel_1493281246' 'ch...,1147,182620,159.215344,0.342063,0.99,0.134368,...,97890.0,27388.596338,0.119034,0.022057,0.003499,1,0.923333,0.857143,High,High
1,cluster_13,13,['channel_1324047869' 'channel_1317428262' 'ch...,['channel_1324047869' 'channel_1317428262' 'ch...,970,156377,161.213402,0.341590,0.99,0.134597,...,97890.0,29331.452577,0.118585,0.023072,0.004368,1,0.846667,0.928571,High,High
2,cluster_15,15,['channel_1500143050' 'channel_1500194629' 'ch...,['channel_1500143050' 'channel_1500194629' 'ch...,1226,181175,147.777325,0.341493,0.99,0.132617,...,98340.0,29902.970636,0.116462,0.023160,0.004371,1,0.820000,1.000000,High,High
3,cluster_16,16,['channel_1658763645' 'channel_1606432449' 'ch...,['channel_1658763645' 'channel_1606432449' 'ch...,1070,162705,152.060748,0.335960,0.99,0.131227,...,98346.0,23816.556075,0.109609,0.022845,0.004757,1,0.413333,0.785714,Medium,High
4,cluster_18,18,['channel_1800847669' 'channel_1808126025' 'ch...,['channel_1800847669' 'channel_1808126025' 'ch...,800,104610,130.762500,0.345614,0.99,0.144066,...,96927.0,14769.757500,0.124520,0.027473,0.011892,0,0.393333,0.571429,Medium,Medium
5,cluster_19,19,['channel_1927028788' 'channel_1964597198' 'ch...,['channel_1927028788' 'channel_1964597198' 'ch...,711,79929,112.417722,0.343001,0.99,0.137669,...,97986.0,14247.611814,0.120457,0.024797,0.008808,0,0.380000,0.500000,Medium,Medium
6,cluster_21,21,['channel_2136263556' 'channel_2140862011' 'ch...,['channel_2136263556' 'channel_2140862011' 'ch...,459,32256,70.274510,0.346426,0.99,0.145339,...,26120.0,5851.043573,0.129061,0.031033,0.012401,0,0.320000,0.285714,Low,Low
7,cluster_22,22,['channel_2234014990' 'channel_2244744109' 'ch...,['channel_2234014990' 'channel_2244744109' 'ch...,115,6049,52.600000,0.347214,0.92,0.131000,...,7452.0,2472.834783,0.125971,0.018350,0.002810,0,0.206667,0.142857,Low,Low
8,cluster_17,17,['channel_1710390801' 'channel_1751710558' 'ch...,['channel_1710390801' 'channel_1751710558' 'ch...,1002,136795,136.521956,0.334055,0.99,0.130880,...,98340.0,16307.973054,0.104258,0.024533,0.005885,0,0.130000,0.642857,Low,Medium
9,cluster_12,12,['channel_1214535459' 'channel_1218420610' 'ch...,['channel_1214535459' 'channel_1218420610' 'ch...,951,140267,147.494217,0.338324,0.98,0.130703,...,46009.0,13459.075710,0.111045,0.021994,0.003828,0,0.123333,0.428571,Low,Medium


Download:
/kaggle/working/telegram_cluster_amplification_outputs.zip


## Ghi chú quan trọng cho báo cáo

Ở cấp cluster group, số dòng train có thể không nhiều vì mỗi cluster là một nhóm lớn. Vì vậy:

- `amplification_score`: score từ RandomForest, dùng cho demo model.
- `observed_amplification_score`: percentile rank trực tiếp theo metric amplification, ổn định hơn khi số cluster ít.
- `forwards_per_channel`: metric khuyến nghị để so sánh công bằng giữa các cluster có số channel khác nhau.
